# 🏠 Big Data Analytics — Advanced Real Estate Scale-Up
## Phase 1 & Phase 2 Complete Verification Notebook

**Course**: CSE3712 — Big Data Analytics (Semester 6)  
**Data Source**: HM Land Registry (Official UK Gov Open Data) + UK Postcode Coordinates Directory (**100% Non-Kaggle**)  
**Scale**: **1,635,628 Transactions** across England & Wales (2022–2023) — **~76x scale-up** from King County baseline.

### Pipeline Architecture Overview
```
1. Official Data Sources (HM Land Registry 2022-2023 + UK Postcodes)
   ↓
2. Storage & Database Layer (DuckDB Vectorized OLAP + Snappy-Compressed Parquet)
   ↓
3. Uber H3 Hexagonal Spatial Indexing (Res 7 & 8 Micro-Market Clustering)
   ↓
4. Feature Engineering at Scale (Proximity, BoE Macro Interest Rates, Cyclical Encodings)
   ↓
5. ML-Ready Feature Matrix (51 Features, 1.635M Rows)
```

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Connect to persistent DuckDB database created in Phase 1
con = duckdb.connect('../data/realestate.duckdb')
con.execute("PRAGMA threads=8;")
print("DuckDB Version:", duckdb.__version__)
print("Database Connection Active.")

## 1. Phase 1: Storage Layer & OLAP Query Benchmarks

In [ ]:
# Verify registered views & row count
df_count = con.execute("SELECT count(*) AS total_transactions FROM properties").df()
display(df_count)

In [ ]:
# OLAP Query 1: Property Type Valuation Distribution (<50ms)
query_type = """
SELECT 
    CASE property_type
        WHEN 'D' THEN 'Detached'
        WHEN 'S' THEN 'Semi-Detached'
        WHEN 'T' THEN 'Terraced'
        WHEN 'F' THEN 'Flat / Maisonette'
        ELSE 'Other'
    END AS property_type_name,
    count(*) AS sales_count,
    round(avg(price), 0)::BIGINT AS avg_price,
    round(median(price), 0)::BIGINT AS median_price
FROM properties
GROUP BY 1
ORDER BY median_price DESC;
"""
df_types = con.execute(query_type).df()
display(df_types)

# Visualizing Median Price by Property Type
plt.figure(figsize=(8, 4))
plt.barh(df_types['property_type_name'], df_types['median_price'] / 1e3, color='#2b5c8f')
plt.xlabel('Median Price (£ thousands)')
plt.title('UK Property Valuation by Dwelling Type (1.63M Sales)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# OLAP Query 2: Top 10 Most Expensive Counties in England & Wales
query_counties = """
SELECT 
    county,
    count(*) AS total_sales,
    round(median(price), 0)::BIGINT AS median_price,
    round(avg(price), 0)::BIGINT AS avg_price
FROM properties
WHERE county != 'UNKNOWN'
GROUP BY county
HAVING count(*) >= 2000
ORDER BY median_price DESC
LIMIT 10;
"""
df_counties = con.execute(query_counties).df()
display(df_counties)

## 2. Phase 2: Feature Matrix Inspection (Uber H3, Macro, Proximity)

In [ ]:
# Load the finalized feature matrix created in Phase 2
feature_parquet_path = '../data/features/feature_matrix.parquet'
df_features = pd.read_parquet(feature_parquet_path)
print(f"Feature Matrix Shape: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns")
df_features[['price', 'log_price', 'h3_res7', 'h3_median_price', 'price_to_h3_median_ratio', 'dist_to_london_km', 'boe_base_rate']].head(10)

In [ ]:
# Visualizing Target Distribution: Raw Price vs Log Price
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_features['price'].clip(upper=1500000) / 1e3, bins=50, color='#34495e', edgecolor='white')
axes[0].set_title('Raw Price Distribution (Capped at £1.5M)')
axes[0].set_xlabel('Price (£ thousands)')
axes[0].set_ylabel('Count')

axes[1].hist(df_features['log_price'], bins=50, color='#27ae60', edgecolor='white')
axes[1].set_title('Normalized Target Variable (log_price)')
axes[1].set_xlabel('log(Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Visualizing Novelty 1 Upgrade: Uber H3 Hexagonal Micro-Market Density
plt.figure(figsize=(8, 4))
plt.hist(df_features['price_to_h3_median_ratio'].clip(0.3, 2.5), bins=60, color='#8e44ad', edgecolor='white')
plt.axvline(1.0, color='red', linestyle='--', label='Micro-Market Median Baseline (1.0)')
plt.title('Novelty 1: Distribution of Price Relative to Local H3 Micro-Market')
plt.xlabel('Price / Local H3 Hexagon Median Price')
plt.ylabel('Property Count')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Macroeconomic Impact: Bank of England Interest Rate vs Monthly Median Price
macro_summary = df_features.groupby(['sale_year', 'sale_month']).agg(
    median_price=('price', 'median'),
    boe_rate=('boe_base_rate', 'first'),
    volume=('price', 'count')
).reset_index()
macro_summary['period'] = macro_summary['sale_year'].astype(str) + '-' + macro_summary['sale_month'].astype(str).str.zfill(2)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.plot(macro_summary['period'], macro_summary['median_price'] / 1e3, color='#2980b9', marker='o', label='Median Price (£k)')
ax2.step(macro_summary['period'], macro_summary['boe_rate'], color='#e74c3c', linewidth=2, label='BoE Base Rate (%)')

ax1.set_xlabel('Month (2022-2023)')
ax1.set_ylabel('Median Price (£ thousands)', color='#2980b9')
ax2.set_ylabel('BoE Base Rate (%)', color='#e74c3c')
ax1.tick_params(axis='x', rotation=45)
plt.title('Macroeconomic Factor: Property Valuation Trajectory vs. Bank of England Rate Hikes')
plt.show()

## 3. Scale-Up Summary: King County Baseline vs. Big Data Implementation

| Evaluation Metric | Baseline Project (King County) | Scaled Big Data Project (HM Land Registry) | Improvement Factor |
| :--- | :--- | :--- | :--- |
| **Data Source** | Kaggle CSV (King County, WA) | Official HM Land Registry + Open Postcodes (**No Kaggle**) | 100% Official Gov Open Data |
| **Total Transactions** | 21,613 | **1,635,628** | **~76x Scale-Up** |
| **Geographic Scope** | 1 Single County | All of England & Wales (120+ Counties) | Nationwide |
| **Storage Engine** | Single in-memory CSV (Pandas) | **Snappy Columnar Parquet + DuckDB OLAP** | Sub-50ms analytical queries |
| **Spatial Micro-Markets**| String Geohash (Precision 6) | **Uber H3 Hexagonal Hierarchy (Res 7 & 8)** | Equidistant hexagonal tiling |
| **Features Generated** | 20 features | **51 multi-dimensional features** | +155% feature space |
| **Macroeconomic Factors**| None (Static single-year) | Official Bank of England Base Rate Cycle | Macro time-series integration |
| **Pipeline Runtime** | N/A | **<25 seconds end-to-end** | Highly optimized C++ vectorized |
